# Sentinel-2 Ürün Sınıflandırması - Faz 2

**Öğrenci:** Esmanur Polattimur  
**Numara:** 241230014

Bu notebookta TUM CropTypes verisiyle Random Forest ve MLP modelleri eğitilmektedir.

In [ ]:
# ============================================================
# 1. KÜTÜPHANELER
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

In [ ]:
# ============================================================
# 2. VERİ SETİNİ OKUMA
# ============================================================

proje_klasoru = Path.cwd()
veri_klasoru = proje_klasoru / "data" / "phase1" / "tum_croptypes"

train_df = pd.read_excel(veri_klasoru / "data2016-2018.xlsx")
test_df = pd.read_excel(veri_klasoru / "TestData.xlsx")

print("Eğitim/geliştirme boyutu:", train_df.shape)
print("Bağımsız test boyutu:", test_df.shape)
print("Eğitim eksik değer:", train_df.isnull().sum().sum())
print("Test eksik değer:", test_df.isnull().sum().sum())

In [ ]:
# ============================================================
# 3. SINIFLARI DÜZENLEME
# ============================================================

kod_degistirme = {
    601: 600, 602: 600, 476: 131, 411: 400,
    171: 400, 410: 400, 177: 400, 489: 311
}

siniflar = {
    600: "patates", 131: "kışlık arpa", 400: "mısır",
    311: "kışlık kolza", 115: "kışlık buğday",
    603: "şeker pancarı", 1: "diğer"
}

sinif_isimleri = list(siniflar.values())
sinif_numaralari = {isim: i for i, isim in enumerate(sinif_isimleri)}
ana_kodlar = [600, 131, 400, 311, 115, 603]

for df in [train_df, test_df]:
    df["mapped_code"] = df["NC"].astype(int).replace(kod_degistirme)
    df.loc[~df["mapped_code"].isin(ana_kodlar), "mapped_code"] = 1
    df["mapped_class"] = df["mapped_code"].map(siniflar)
    df["target"] = df["mapped_class"].map(sinif_numaralari)

print("Eğitim sınıfları:")
print(train_df["mapped_class"].value_counts())
print("\nTest sınıfları:")
print(test_df["mapped_class"].value_counts())

In [ ]:
# ============================================================
# 4. X VE y OLUŞTURMA
# ============================================================

ozellik_kolonlari = [
    kolon for kolon in train_df.columns
    if str(kolon).endswith("_mean") and kolon in test_df.columns
]

X = train_df[ozellik_kolonlari]
y = train_df["target"]

X_test = test_df[ozellik_kolonlari]
y_test = test_df["target"]

print("Özellik sayısı:", len(ozellik_kolonlari))
print("X boyutu:", X.shape)
print("y boyutu:", y.shape)

In [ ]:
# ============================================================
# 5. EĞİTİM / DOĞRULAMA BÖLMESİ
# ============================================================

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print("X_train:", X_train.shape)
print("X_valid:", X_valid.shape)
print("X_test :", X_test.shape)

In [ ]:
# ============================================================
# 6. EKSİK DEĞERLERİ DOLDURMA
# ============================================================

imputer = SimpleImputer(strategy="median")

X_train_dolu = imputer.fit_transform(X_train)
X_valid_dolu = imputer.transform(X_valid)
X_test_dolu = imputer.transform(X_test)

print("Train eksik:", np.isnan(X_train_dolu).sum())
print("Valid eksik:", np.isnan(X_valid_dolu).sum())
print("Test eksik :", np.isnan(X_test_dolu).sum())

In [ ]:
# ============================================================
# 7. RANDOM FOREST MODELİ
# ============================================================

rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=20,
    class_weight="balanced",
    random_state=42,
    n_jobs=1
)

rf_model.fit(X_train_dolu, y_train)

rf_valid_tahmin = rf_model.predict(X_valid_dolu)
rf_test_tahmin = rf_model.predict(X_test_dolu)

In [ ]:
# ============================================================
# 8. RANDOM FOREST SONUÇLARI
# ============================================================

print("--- RANDOM FOREST - DOĞRULAMA ---")
print("Accuracy:", accuracy_score(y_valid, rf_valid_tahmin))
print("Macro-F1:", f1_score(y_valid, rf_valid_tahmin, labels=list(range(7)), average="macro", zero_division=0))
print("Balanced Accuracy:", balanced_accuracy_score(y_valid, rf_valid_tahmin))
print(classification_report(
    y_valid, rf_valid_tahmin,
    labels=list(range(7)), target_names=sinif_isimleri, zero_division=0
))

print("--- RANDOM FOREST - BAĞIMSIZ TEST ---")
print("Accuracy:", accuracy_score(y_test, rf_test_tahmin))
print("Macro-F1:", f1_score(y_test, rf_test_tahmin, labels=list(range(7)), average="macro", zero_division=0))
print("Balanced Accuracy:", balanced_accuracy_score(y_test, rf_test_tahmin))
print(classification_report(
    y_test, rf_test_tahmin,
    labels=list(range(7)), target_names=sinif_isimleri, zero_division=0
))

In [ ]:
# Confusion matrix
rf_matris = confusion_matrix(y_test, rf_test_tahmin, labels=list(range(7)))

ConfusionMatrixDisplay(
    confusion_matrix=rf_matris, display_labels=sinif_isimleri
).plot(cmap="Blues", xticks_rotation=35)

plt.title("Random Forest - Bağımsız Test")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 9. MLP İÇİN VERİYİ STANDARTLAŞTIRMA
# ============================================================

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_dolu)
X_valid_scaled = scaler.transform(X_valid_dolu)
X_test_scaled = scaler.transform(X_test_dolu)

In [ ]:
# ============================================================
# 10. MLP MODELİ
# ============================================================

mlp_model = MLPClassifier(
    hidden_layer_sizes=(128, 64),
    activation="relu",
    max_iter=250,
    early_stopping=True,
    validation_fraction=0.15,
    random_state=42
)

mlp_model.fit(X_train_scaled, y_train)

mlp_valid_tahmin = mlp_model.predict(X_valid_scaled)
mlp_test_tahmin = mlp_model.predict(X_test_scaled)

In [ ]:
# ============================================================
# 11. MLP SONUÇLARI
# ============================================================

print("--- MLP - DOĞRULAMA ---")
print("Accuracy:", accuracy_score(y_valid, mlp_valid_tahmin))
print("Macro-F1:", f1_score(y_valid, mlp_valid_tahmin, labels=list(range(7)), average="macro", zero_division=0))
print("Balanced Accuracy:", balanced_accuracy_score(y_valid, mlp_valid_tahmin))

print("--- MLP - BAĞIMSIZ TEST ---")
print("Accuracy:", accuracy_score(y_test, mlp_test_tahmin))
print("Macro-F1:", f1_score(y_test, mlp_test_tahmin, labels=list(range(7)), average="macro", zero_division=0))
print("Balanced Accuracy:", balanced_accuracy_score(y_test, mlp_test_tahmin))
print(classification_report(
    y_test, mlp_test_tahmin,
    labels=list(range(7)), target_names=sinif_isimleri, zero_division=0
))

## Sonraki adımlar

1. Random Forest ve MLP sonuçlarını karşılaştırmak.
2. XGBoost modelini eklemek.
3. Zaman boyutunu kullanan 1D-CNN modelini eklemek.
4. Aynı ölçümleri BreizhCrops veri setinde uygulamak.